In [1]:
# 03b_guardrail_agent_llm_raw.ipynb
# =============================================================================
# Notebook 03b — Guardrail Agent with LLM Raw Range Capture
#
# Purpose (revision, Reviewer A #4 & #5):
#   The original notebook 03 stored only the LLM's natural-language reasoning
#   and derived all final ranges from Hard Rules A-E. To evaluate the LLM's
#   *own* proposed numeric ranges (the "LLM-only" condition in Table 4/5/6)
#   and to enable a clean Hard-Rule-only ablation, this notebook captures and
#   stores THREE range sets per case:
#       (1) llm_raw_ranges   — parsed directly from the LLM JSON output
#       (2) hardrule_ranges  — Hard Rules A-E from patient baseline (no LLM)
#       (3) final_ranges     — LLM raw ranges THEN Hard Rule post-processing
#
#   The LLM call itself (prompt, system message, API parameters) is UNCHANGED
#   from notebook 03.
#
# Input  : ../results/tables/agent_config.pkl
#          ../results/tables/df_final.pkl
#          ../results/tables/model_{group}.pkl
# Output : ../results/tables/guardrail_ranges_v2.json
# =============================================================================

# %%
# ## 0. Import Libraries
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# %%
# ## 1. Load Models and Configuration
agent_config = joblib.load('../results/tables/agent_config.pkl')
df_final     = joblib.load('../results/tables/df_final.pkl')

X_FEATURES      = agent_config['X_features']
NUM_FEATURES    = agent_config['num_features']
CAT_FEATURES    = agent_config['cat_features']
VARY_FEATURES   = agent_config['vary_features']
FIXED_FEATURES  = agent_config['fixed_features']
TARGET_COL      = agent_config['target_col']
AGEGROUP_CONFIG = agent_config['agegroup_config']

models = {}
for group_name in AGEGROUP_CONFIG:
    try:
        models[group_name] = joblib.load(
            f'../results/tables/model_{group_name}.pkl'
        )
    except FileNotFoundError:
        print(f"WARNING: model_{group_name}.pkl not found.")
        models[group_name] = None

print("Models loaded:", [k for k, v in models.items() if v is not None])

# %%
# ## 2. OpenAI Client Setup
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL', 'gpt-4o-mini')

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"LLM model: {LLM_MODEL}")

# %%
# ## 3. Representative Case Index Selection  (unchanged from nb03)
CASE_INDICES = {
    "MiddleAged_Male":   [3972, 5762, 4265],
    "MiddleAged_Female": [5903, 1561, 5323],
    "Older_Male":        [4793, 1673, 4989],
    "Older_Female":      [4918, 3010, 3991],
}

# %%
# ## 4. LLM-Based Clinical Guardrail Agent  (UNCHANGED from nb03)
def get_clinical_guardrails(patient_data: pd.DataFrame,
                             group_name: str,
                             allowed_features: list,
                             metadata: dict) -> dict:
    """
    Call GPT-4o-mini. Prompt, system message and API parameters are IDENTICAL
    to notebook 03. Returns the parsed JSON dict, which may contain the keys
    'reasoning' and 'guardrail_ranges'.
    """
    patient_info = patient_data.to_dict(orient='records')[0]

    prompt = f"""
You are a high-precision Clinical Data Strategist for a health insurance company.
You must respond ONLY in valid JSON using the exact variable names listed below.

[Patient Information ({group_name})]
{json.dumps(patient_info, ensure_ascii=False, indent=1)}

[Available Variable Names]
{", ".join(allowed_features)}

[Categorical Variable Valid Values]
{json.dumps(metadata, ensure_ascii=False, indent=1)}

[Guardrail Principles — apply ALL strictly]

1. CONFLICT PREVENTION:
   - Cap Carb_g at [0, current Carb_g] and Sugar_g at [0, current Sugar_g]
     whenever Sodium_mg is reduced below its current value.

2. PHYSICAL CONSISTENCY:
   - BMI, WaistCirc, Weight: all must move in the same direction.
   - Each bounded to [current * 0.85, current].

3. ENERGY FLOOR:
   - Energy_kcal: [max(current * 0.70, 500), current].

4. SAFETY: All lower bounds >= 0.

5. DIVERSITY HINT:
   - Allow Fat_g to vary in both directions for structural pathway diversity.

[Response Format — valid JSON only, no extra text]
{{
    "reasoning": "Detailed clinical rationale for each constraint",
    "guardrail_ranges": {{ "ExactVariableName": [min, max] }}
}}
"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert who responds strictly in valid JSON, "
                    "fully adhering to clinical guidelines and the provided "
                    "data schema."
                )
            },
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

# %%
# ## 5. Parse LLM Raw Ranges  (NEW)
def parse_llm_raw_ranges(agent_plan: dict, features: list) -> dict:
    """
    Extract the LLM's proposed numeric [min, max] ranges from its JSON output,
    WITHOUT applying any hard rule. This is the 'LLM-only' condition.

    The LLM output is not schema-guaranteed: the 'guardrail_ranges' block may
    be missing, partially filled, or wrapped in nested dicts. We extract only
    entries that are unambiguously a numeric [min, max] pair for a known
    feature. Everything else is left ABSENT (not imputed) so that downstream
    evaluation reflects the LLM's real coverage.

    Returns
    -------
    dict: {feature: [min, max]} containing ONLY features the LLM supplied as
          a valid numeric pair. Missing features are intentionally omitted.
    """
    raw = agent_plan.get('guardrail_ranges', {})
    parsed = {}

    if not isinstance(raw, dict):
        return parsed

    for feat in features:
        if feat not in raw:
            continue
        val = raw[feat]
        # Accept only a clean 2-element numeric pair.
        if (isinstance(val, (list, tuple)) and len(val) == 2
                and all(isinstance(x, (int, float)) for x in val)):
            lo, hi = float(val[0]), float(val[1])
            if lo <= hi:
                parsed[feat] = [round(lo, 4), round(hi, 4)]
            else:
                parsed[feat] = [round(hi, 4), round(lo, 4)]
        # else: nested/malformed -> treat as NOT supplied (omit)
    return parsed

# %%
# ## 6. Hard Rules A-E  (UNCHANGED from nb03; no LLM input)
def apply_hard_rules(query: pd.DataFrame, features: list) -> dict:
    cur = {f: float(query.iloc[0][f]) for f in features}
    g   = {}
    for f in features:
        g[f] = [round(max(cur[f] * 0.70, 0.0), 4),
                round(cur[f] * 1.30, 4)]
    for f in ['BMI', 'WaistCirc', 'Weight']:
        g[f] = [round(cur[f] * 0.85, 4), round(cur[f], 4)]
    wt_ratio = g['Weight'][0] / cur['Weight']
    g['WaistCirc'][0] = max(g['WaistCirc'][0],
                            round(cur['WaistCirc'] * wt_ratio, 4))
    g['BMI'][0]       = max(g['BMI'][0],
                            round(cur['BMI']       * wt_ratio, 4))
    g['Energy_kcal']  = [round(max(cur['Energy_kcal'] * 0.70, 500.0), 4),
                         round(cur['Energy_kcal'], 4)]
    g['Sodium_mg']    = [round(max(cur['Sodium_mg'] * 0.60, 800.0), 4),
                         round(cur['Sodium_mg'] * 0.90, 4)]
    g['Carb_g'][1]    = round(cur['Carb_g'],  4)
    g['Sugar_g'][1]   = round(cur['Sugar_g'], 4)
    g['Protein_g'][0]    = round(max(cur['Protein_g']    * 0.60, 30.0),  4)
    g['Potassium_mg'][0] = round(max(cur['Potassium_mg'] * 0.50, 500.0), 4)
    g['Carb_g'][0]       = round(max(cur['Carb_g']       * 0.20, 30.0),  4)
    g['Fiber_g'][0]      = round(max(cur['Fiber_g']       * 0.30, 5.0),  4)
    for f in features:
        if g[f][0] > g[f][1]:
            g[f] = [round(cur[f] * 0.85, 4), round(cur[f], 4)]
    return g

# %%
# ## 7. LLM Raw + Hard Rule Post-Processing  (NEW)
def apply_hardrule_postprocess(llm_raw: dict,
                                query: pd.DataFrame,
                                features: list) -> dict:
    """
    The 'LLM + Hard Rule' condition: start from the LLM's raw ranges, then
    enforce Hard Rules A-E on top. For any feature the LLM did not supply,
    fall back to the hard-rule range for that feature (this is exactly the
    safety-net behaviour the paper describes: the deterministic layer fills
    gaps left by incomplete LLM output).
    """
    hard = apply_hard_rules(query, features)
    cur  = {f: float(query.iloc[0][f]) for f in features}
    out  = {}

    for f in features:
        if f in llm_raw:
            lo, hi = llm_raw[f]
        else:
            # LLM did not supply -> hard rule fills the gap
            out[f] = list(hard[f])
            continue
        # Intersect LLM proposal with hard-rule bounds (hard rule wins on conflict)
        h_lo, h_hi = hard[f]
        new_lo = max(lo, h_lo)
        new_hi = min(hi, h_hi)
        if new_lo > new_hi:
            # LLM proposal incompatible with hard rule -> hard rule governs
            new_lo, new_hi = h_lo, h_hi
        out[f] = [round(new_lo, 4), round(new_hi, 4)]
    return out

# %%
# ## 8. Generate ALL THREE Range Sets for 12 Cases
all_guardrails = {}
feature_metadata = {
    col: sorted(df_final[col].unique().tolist())
    for col in CAT_FEATURES if col in df_final.columns
}

for group_name, cfg in AGEGROUP_CONFIG.items():
    mdl = models.get(group_name)
    if mdl is None:
        print(f"\n[{group_name}] Model not available. Skipping.")
        continue

    for case_num, idx in enumerate(CASE_INDICES[group_name], start=1):
        case_key = f"{group_name}_case{case_num}"
        print(f"\n{'='*15} [{case_key}] {'='*15}")

        query = df_final.loc[[idx], X_FEATURES].copy()

        # --- LLM call (UNCHANGED) ---
        print("Calling GPT-4o-mini guardrail agent...")
        agent_plan = get_clinical_guardrails(
            query, group_name, X_FEATURES, feature_metadata
        )
        if isinstance(agent_plan, list):
            agent_plan = agent_plan[0] if agent_plan else {}
        reasoning = str(agent_plan.get('reasoning', ''))

        # --- Three range sets ---
        llm_raw   = parse_llm_raw_ranges(agent_plan, X_FEATURES)
        hardrule  = apply_hard_rules(query, X_FEATURES)
        final     = apply_hardrule_postprocess(llm_raw, query, X_FEATURES)

        n_supplied = len(llm_raw)
        print(f"  LLM supplied {n_supplied}/{len(X_FEATURES)} "
              f"valid numeric ranges")
        missing = [f for f in X_FEATURES if f not in llm_raw]
        if missing:
            print(f"  LLM missing features: {missing}")

        all_guardrails[case_key] = {
            'group':            group_name,
            'case':             case_num,
            'original_index':   idx,
            'patient_profile':  query.to_dict(orient='records')[0],
            'llm_reasoning':    reasoning,
            'llm_raw_ranges':   llm_raw,     # (1) LLM-only  [may be partial]
            'hardrule_ranges':  hardrule,    # (2) HardRule-only
            'final_ranges':     final,       # (3) LLM + HardRule
            'llm_n_supplied':   n_supplied,
            'llm_missing':      missing,
        }

# %%
# ## 9. Save
with open('../results/tables/guardrail_ranges_v2.json', 'w',
          encoding='utf-8') as f:
    json.dump(all_guardrails, f, ensure_ascii=False, indent=4)

print(f"\n>>> Saved: ../results/tables/guardrail_ranges_v2.json")
print(f"    Total cases: {len(all_guardrails)}")
print("\n  LLM range coverage per case:")
for k, v in all_guardrails.items():
    print(f"    {k:28s}: {v['llm_n_supplied']}/{len(X_FEATURES)} supplied")

Models loaded: ['MiddleAged_Male', 'MiddleAged_Female', 'Older_Male', 'Older_Female']
LLM model: gpt-4o-mini

=============== [MiddleAged_Male_case1] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied 31/31 valid numeric ranges

=============== [MiddleAged_Male_case2] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied 31/31 valid numeric ranges

=============== [MiddleAged_Male_case3] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied 31/31 valid numeric ranges

=============== [MiddleAged_Female_case1] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied 31/31 valid numeric ranges

=============== [MiddleAged_Female_case2] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied 7/31 valid numeric ranges
  LLM missing features: ['Sodium_mg', 'SaturatedFat_g', 'Fiber_g', 'Potassium_mg', 'Protein_g', 'ObesityStatus', 'WeightChangeStatus', 'WeightLossAmount', 'WeightGainAmount', 'DrinkingFrequency', 'D

In [2]:
# =============================================================================
# 03b — REPLACEMENT for Section 5 (parsing) and Sections 8-9 (generate/save)
# Keep Sections 0-4 (imports, config, OpenAI client, LLM call) and
# Sections 6-7 (apply_hard_rules, apply_hardrule_postprocess) UNCHANGED.
# =============================================================================

# %%
# ## 5. Parse LLM Raw Ranges — TWO policies (NEW)

def _clean_pair(val):
    """Return [lo, hi] if val is a clean 2-element numeric pair, else None."""
    if isinstance(val, (list, tuple)) and len(val) == 2 \
            and all(isinstance(x, (int, float)) and not isinstance(x, bool)
                    for x in val):
        lo, hi = float(val[0]), float(val[1])
        return [round(min(lo, hi), 4), round(max(lo, hi), 4)]
    return None


def parse_llm_raw_strict(agent_plan: dict, features: list) -> dict:
    """
    STRICT policy: accept a feature ONLY if guardrail_ranges[feature] is a
    clean numeric [min, max] pair. Nested dicts / prose / missing keys are
    treated as NOT supplied (feature omitted). This reflects the LLM's real
    machine-usable coverage.
    """
    raw = agent_plan.get('guardrail_ranges', {})
    parsed = {}
    if not isinstance(raw, dict):
        return parsed
    for feat in features:
        if feat in raw:
            pair = _clean_pair(raw[feat])
            if pair is not None:
                parsed[feat] = pair
    return parsed


def parse_llm_raw_lenient(agent_plan: dict, features: list) -> dict:
    """
    LENIENT policy: in addition to clean pairs, attempt to recover a [min, max]
    from nested dicts. Recognised patterns inside a per-feature dict:
        {'min': x, 'max': y}
        {'lower': x, 'upper': y}
        {'current': c, 'min': x, 'max': y}   -> uses min/max
    Also searches one level of nesting under top-level principle blocks
    (e.g. CONFLICT_PREVENTION / PHYSICAL_CONSISTENCY) in case the LLM nested
    feature ranges under a principle heading. Values must be numeric.
    """
    raw = agent_plan.get('guardrail_ranges', {})
    parsed = {}

    def try_dict_pair(d):
        if not isinstance(d, dict):
            return None
        keysets = [('min', 'max'), ('lower', 'upper'),
                   ('Min', 'Max'), ('low', 'high')]
        for k_lo, k_hi in keysets:
            if k_lo in d and k_hi in d:
                lo, hi = d[k_lo], d[k_hi]
                if all(isinstance(x, (int, float)) and not isinstance(x, bool)
                       for x in (lo, hi)):
                    return [round(min(lo, hi), 4), round(max(lo, hi), 4)]
        return None

    if not isinstance(raw, dict):
        return parsed

    # Pass 1: direct feature entries (clean pair OR recoverable dict)
    for feat in features:
        if feat in raw:
            pair = _clean_pair(raw[feat])
            if pair is None:
                pair = try_dict_pair(raw[feat])
            if pair is not None:
                parsed[feat] = pair

    # Pass 2: one level of nesting (principle-block -> feature -> pair/dict)
    for block_val in raw.values():
        if isinstance(block_val, dict):
            for feat in features:
                if feat in parsed:
                    continue
                if feat in block_val:
                    pair = _clean_pair(block_val[feat])
                    if pair is None:
                        pair = try_dict_pair(block_val[feat])
                    if pair is not None:
                        parsed[feat] = pair
    return parsed

# %%
# ## 8. Generate ALL range sets for 12 cases (REPLACEMENT)

all_guardrails = {}
feature_metadata = {
    col: sorted(df_final[col].unique().tolist())
    for col in CAT_FEATURES if col in df_final.columns
}

for group_name, cfg in AGEGROUP_CONFIG.items():
    mdl = models.get(group_name)
    if mdl is None:
        print(f"\n[{group_name}] Model not available. Skipping.")
        continue

    for case_num, idx in enumerate(CASE_INDICES[group_name], start=1):
        case_key = f"{group_name}_case{case_num}"
        print(f"\n{'='*15} [{case_key}] {'='*15}")

        query = df_final.loc[[idx], X_FEATURES].copy()

        print("Calling GPT-4o-mini guardrail agent...")
        agent_plan = get_clinical_guardrails(
            query, group_name, X_FEATURES, feature_metadata
        )
        if isinstance(agent_plan, list):
            agent_plan = agent_plan[0] if agent_plan else {}
        reasoning = str(agent_plan.get('reasoning', ''))

        # Two LLM-only parses
        llm_strict  = parse_llm_raw_strict(agent_plan, X_FEATURES)
        llm_lenient = parse_llm_raw_lenient(agent_plan, X_FEATURES)

        # Hard rule only
        hardrule = apply_hard_rules(query, X_FEATURES)

        # LLM + hard rule (use strict parse as the LLM contribution;
        # gaps filled by hard rule — matches paper's described pipeline)
        final = apply_hardrule_postprocess(llm_strict, query, X_FEATURES)

        n_strict  = len(llm_strict)
        n_lenient = len(llm_lenient)
        print(f"  LLM supplied (strict) : {n_strict}/{len(X_FEATURES)}")
        print(f"  LLM supplied (lenient): {n_lenient}/{len(X_FEATURES)}")

        all_guardrails[case_key] = {
            'group':               group_name,
            'case':                case_num,
            'original_index':      idx,
            'patient_profile':     query.to_dict(orient='records')[0],
            'llm_reasoning':       reasoning,
            'llm_raw_strict':      llm_strict,    # LLM-only (strict)
            'llm_raw_lenient':     llm_lenient,   # LLM-only (lenient)
            'hardrule_ranges':     hardrule,      # HardRule-only
            'final_ranges':        final,         # LLM + HardRule
            'llm_n_strict':        n_strict,
            'llm_n_lenient':       n_lenient,
            'llm_missing_strict':  [f for f in X_FEATURES
                                    if f not in llm_strict],
        }

# %%
# ## 9. Save (REPLACEMENT)
with open('../results/tables/guardrail_ranges_v2.json', 'w',
          encoding='utf-8') as f:
    json.dump(all_guardrails, f, ensure_ascii=False, indent=4)

print(f"\n>>> Saved: ../results/tables/guardrail_ranges_v2.json")
print("\n  LLM range coverage per case (strict / lenient):")
for k, v in all_guardrails.items():
    print(f"    {k:28s}: {v['llm_n_strict']:2d} / "
          f"{v['llm_n_lenient']:2d}  of {len(X_FEATURES)}")


=============== [MiddleAged_Male_case1] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied (strict) : 31/31
  LLM supplied (lenient): 31/31

=============== [MiddleAged_Male_case2] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied (strict) : 31/31
  LLM supplied (lenient): 31/31

=============== [MiddleAged_Male_case3] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied (strict) : 31/31
  LLM supplied (lenient): 31/31

=============== [MiddleAged_Female_case1] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied (strict) : 31/31
  LLM supplied (lenient): 31/31

=============== [MiddleAged_Female_case2] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied (strict) : 31/31
  LLM supplied (lenient): 31/31

=============== [MiddleAged_Female_case3] ===============
Calling GPT-4o-mini guardrail agent...
  LLM supplied (strict) : 31/31
  LLM supplied (lenient): 31/31

=============== [Older_Male_case1